# scGPT RNA Embedding & Spatial Clustering (COSMOS Mouse Brain Dataset)
This notebook integrates **scGPT** high-dimensional RNA embeddings with low-dimensional PCA features of RNA (omic1) and ATAC features (omic2) on the COSMOS Mouse Brain dataset.
It builds a K-Nearest Neighbors (KNN) cell graph on the fused representation and performs clustering using KMeans and Leiden algorithms, evaluating their performance against ground truth cortical/brain layer annotations.

### Kaggle Requirements:
1. **scGPT Model Dataset** (e.g., `scgpt-human` containing `scGPT_human` weight folder).
2. **COSMOS Dataset** (e.g., `cosmos-data` containing `ATAC_RNA_Seq_MouseBrain_RNA_ATAC.h5`).
3. **GPU Accelerator** enabled (T4 or P100) for running scGPT embedding generation.

> [!IMPORTANT]
> **Kernel Restart Note:** Running the Environment Setup cell below will automatically restart the Jupyter notebook kernel to apply the package downgrades (like numpy/scipy) and avoid any `ModuleNotFoundError` or `numpy.rec` conflicts. This is expected behavior! After the kernel restarts, you can run the rest of the cells sequentially.

In [ ]:
# 1. Environment Setup (Kaggle & Colab compatible)
# Force install PyPI torch and torchtext compatible versions first
!pip install -q torch==2.3.0 torchvision==0.18.0 torchaudio==2.3.0 torchtext==0.18.0

# Uninstall any conflicting numpy/scipy packages
!pip uninstall -y numpy scipy scikit-learn pandas scanpy anndata datasets
!rm -rf /usr/local/lib/python3.12/dist-packages/numpy*
!rm -rf /usr/local/lib/python3.12/dist-packages/scipy*

# Install strictly compatible Spring 2024 Stack versions
!pip install -q "numpy==1.26.4" "scipy==1.13.1" "scikit-learn==1.4.2" "pandas==2.2.2"

# Install omics and graph clustering packages (using python-igraph for fast and compile-free Leiden clustering)
!pip install -q scanpy anndata datasets h5py python-igraph

# Install scGPT library
!pip install -q scgpt==0.2.4 --no-deps

# Programmatically restart the kernel to load the new packages without any import errors (e.g., numpy.rec)
import os
print("Environment setup complete. Restarting kernel...")
os.kill(os.getpid(), 9)

In [ ]:
# 2. Imports and Seed Initialization
import os
import random
import h5py
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
from torch.backends import cudnn
import sklearn
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from scipy.sparse import coo_matrix
import anndata as ad
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import Optional
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    v_measure_score,
    silhouette_score
)

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def fix_seed(seed: int = 2026):
    """Fix random seed for reproducibility."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = True
    cudnn.benchmark = False

fix_seed(2026)

In [ ]:
# 3. Data Loading
# Toggle directories automatically for Kaggle vs Local environments
KAGGLE_DATA_PATH = '/kaggle/input/cosmos-data/ATAC_RNA_Seq_MouseBrain_RNA_ATAC.h5'
LOCAL_DATA_PATH = '../../spaLLM/spaLLM/DATA_COSMOS/ATAC_RNA_Seq_MouseBrain_RNA_ATAC.h5'
ALT_LOCAL_DATA_PATH = '../spaLLM/spaLLM/DATA_COSMOS/ATAC_RNA_Seq_MouseBrain_RNA_ATAC.h5'
ALT2_LOCAL_DATA_PATH = 'd:/FYDP/spaLLM/spaLLM/DATA_COSMOS/ATAC_RNA_Seq_MouseBrain_RNA_ATAC.h5'

if os.path.exists(KAGGLE_DATA_PATH):
    file_path = KAGGLE_DATA_PATH
elif os.path.exists(LOCAL_DATA_PATH):
    file_path = LOCAL_DATA_PATH
elif os.path.exists(ALT_LOCAL_DATA_PATH):
    file_path = ALT_LOCAL_DATA_PATH
elif os.path.exists(ALT2_LOCAL_DATA_PATH):
    file_path = ALT2_LOCAL_DATA_PATH
else:
    raise FileNotFoundError("COSMOS dataset H5 file not found in paths checked.")

print(f"Loading data from {file_path}...")
with h5py.File(file_path, 'r') as h5_f:
    x_rna = h5_f['X_RNA'][()]
    x_atac = h5_f['X_ATAC'][()]
    cell = h5_f['Cell'][()]
    gene = h5_f['Gene'][()]
    pos = h5_f['Pos'][()]
    layer = h5_f['LayerName'][()]

# Decode byte strings if necessary
cell_names = [c.decode('utf-8') if isinstance(c, bytes) else c for c in cell]
gene_names = [g.decode('utf-8') if isinstance(g, bytes) else g for g in gene]
layers = [l.decode('utf-8') if isinstance(l, bytes) else l for l in layer]

# Create AnnData objects (omitting explicit dtype for backward compatibility with older anndata versions)
adata_rna = ad.AnnData(X=x_rna)
adata_rna.obs_names = cell_names
adata_rna.var_names = gene_names
adata_rna.obsm['spatial'] = pos
adata_rna.obs['ground_truth'] = layers

adata_atac = ad.AnnData(X=x_atac)
adata_atac.obs_names = cell_names
adata_atac.obsm['spatial'] = pos

print("RNA shape:", adata_rna.shape)
print("ATAC shape:", adata_atac.shape)

In [ ]:
# 4. scGPT Embedding Generation
from scgpt.tasks.cell_emb import embed_data

KAGGLE_MODEL_DIR = '/kaggle/input/datasets/sadmanbiazidarnob/scgpt-human/scGPT_human'
LOCAL_MODEL_DIR = 'scGPT_human'
ALT_LOCAL_MODEL_DIR = '../../scGPT_human'
ALT2_LOCAL_MODEL_DIR = '../scGPT_human'

if os.path.exists(KAGGLE_MODEL_DIR):
    model_dir = KAGGLE_MODEL_DIR
elif os.path.exists(LOCAL_MODEL_DIR):
    model_dir = LOCAL_MODEL_DIR
elif os.path.exists(ALT_LOCAL_MODEL_DIR):
    model_dir = ALT_LOCAL_MODEL_DIR
elif os.path.exists(ALT2_LOCAL_MODEL_DIR):
    model_dir = ALT2_LOCAL_MODEL_DIR
else:
    model_dir = LOCAL_MODEL_DIR

adata_rna.var_names_make_unique()
adata_rna.var['gene_names'] = adata_rna.var.index.str.upper()

# scGPT expects sparse, non-negative expression inputs.
# Since the COSMOS RNA matrix is pre-scaled (z-scored), it is completely dense (containing negative values).
# We re-introduce sparsity and restore non-negativity by thresholding negative values (which represent expressions below average) to 0.0.
if np.min(adata_rna.X) < 0:
    print("Detected negative values (z-scored RNA matrix). Thresholding values < 0.0 to 0.0 to restore sparsity for scGPT...")
    adata_rna_for_emb = adata_rna.copy()
    # Mask negative values as exactly zero
    adata_rna_for_emb.X[adata_rna_for_emb.X < 0.0] = 0.0
else:
    adata_rna_for_emb = adata_rna.copy()

print("Running scGPT embedding...")
if os.path.exists(model_dir):
    try:
        adata_emb = embed_data(
            adata_or_file=adata_rna_for_emb, 
            model_dir=model_dir, 
            gene_col="gene_names", 
            max_length=1200, 
            batch_size=64, 
            obs_to_save=None, 
            device=device, 
            use_fast_transformer=False, 
            return_new_adata=False
        )
        X_scGPT = adata_emb.obsm["X_scGPT"]
        print("scGPT embedding generated successfully.")
    except Exception as e:
        print(f"Error running embed_data: {e}. Generating dummy embeddings for fallback.")
        X_scGPT = np.random.normal(size=(adata_rna.n_obs, 512))
else:
    print(f"Warning: Model path '{model_dir}' not found. Generating dummy random embeddings for local testing/validation.")
    X_scGPT = np.random.normal(size=(adata_rna.n_obs, 512))

adata_rna.obsm['X_scGPT'] = X_scGPT
print("scGPT matrix shape:", X_scGPT.shape)

In [ ]:
# 5. Preprocessing Omics Data (Omic1 & Omic2)
# Since the COSMOS dataset is already normalized and scaled, we perform PCA directly on the RNA matrix
# to reduce dimensionality, and we use the precomputed ATAC features (which are already low-dimensional PCA features).

def pca(adata: ad.AnnData, use_reps=None, n_comps=50):
    """Perform PCA for dimensionality reduction."""
    pca_model = PCA(n_components=n_comps)
    data = adata.obsm[use_reps] if use_reps else adata.X
    data = data.toarray() if sp.issparse(data) else data
    return pca_model.fit_transform(data)

print("Preprocessing Omic 1 (RNA)...")
# Reduce RNA to 50 dimensions using PCA
adata_rna.obsm['feat'] = pca(adata_rna, n_comps=50)
print(f"RNA PCA feature representation shape: {adata_rna.obsm['feat'].shape}")

print("Preprocessing Omic 2 (ATAC)...")
# Since ATAC features are already 50 dimensions, we use them directly as PCA features
adata_atac.obsm['feat'] = adata_atac.X
print(f"ATAC feature representation shape: {adata_atac.obsm['feat'].shape}")

In [ ]:
# 6. Concatenate scGPT Embedding with Omic1 and Omic2
X_scGPT = adata_rna.obsm['X_scGPT']
feat_rna = adata_rna.obsm['feat']
feat_atac = adata_atac.obsm['feat']

# Fusing scGPT embedding matrix with RNA and ATAC features
joint_feat = np.concatenate((X_scGPT, feat_rna, feat_atac), axis=1)
adata_rna.obsm['joint_feat'] = joint_feat
print("Concatenated (fused) feature matrix shape:", joint_feat.shape)

In [ ]:
# 7. KNN Graph Construction and Spatial Clustering
print("Constructing KNN graph on joint features...")
sc.pp.neighbors(adata_rna, use_rep='joint_feat', n_neighbors=10)
print("KNN graph computed successfully.")

# Determine target number of clusters from ground truth
valid_labels = adata_rna.obs['ground_truth'].dropna().unique()
target_labels = [l for l in valid_labels if l not in ['Exclude', 'unknown', '-1']]
n_clusters = len(target_labels) if len(target_labels) > 0 else 9
print(f"Target cluster count: {n_clusters}")

# 1. KMeans Clustering
print("Running KMeans...")
k_means_model = KMeans(n_clusters=n_clusters, random_state=0, n_init=10)
adata_rna.obs['kmeans_clusters'] = k_means_model.fit_predict(joint_feat).astype(str)

# 2. Leiden Clustering on the KNN Graph
print("Running Leiden resolution search...")
def search_res(adata, target_k, start=0.1, end=3.0, increment=0.05):
    for res in np.arange(start, end, increment):
        res = round(res, 3)
        sc.tl.leiden(adata, random_state=0, resolution=res, key_added='temp_leiden', flavor='igraph', n_iterations=2, directed=False)
        unique_clusters = adata.obs['temp_leiden'].nunique()
        print(f"Resolution: {res} -> Clusters: {unique_clusters}")
        if unique_clusters == target_k:
            return res
    return 0.5

best_res = search_res(adata_rna, n_clusters)
sc.tl.leiden(adata_rna, random_state=0, resolution=best_res, key_added='leiden_clusters', flavor='igraph', n_iterations=2, directed=False)
adata_rna.obs['leiden_clusters'] = adata_rna.obs['leiden_clusters'].astype(str)
print(f"Leiden clustering finished with resolution={best_res}")

In [ ]:
# 8. Cluster Performance Evaluation
def evaluate_clustering(y_true_series, y_pred_series, features_matrix, name=""):
    # Filter out Exclude/unknown/-1 values from verification metrics
    mask = (y_true_series != 'Exclude') & (y_true_series != 'unknown') & (y_true_series != '-1')
    y_true = y_true_series[mask].astype(str)
    y_pred = y_pred_series[mask].astype(str)
    feats = features_matrix[mask]
    
    ari = adjusted_rand_score(y_true, y_pred)
    nmi = normalized_mutual_info_score(y_true, y_pred)
    ami = adjusted_mutual_info_score(y_true, y_pred)
    homogeneity = homogeneity_score(y_true, y_pred)
    v_measure = v_measure_score(y_true, y_pred)
    sil = silhouette_score(feats, y_pred.astype(int) if y_pred.str.isdigit().all() else pd.factorize(y_pred)[0])
    
    print(f"\n=== {name} Clustering Performance ===")
    print(f"ARI: {ari:.4f}")
    print(f"NMI: {nmi:.4f}")
    print(f"AMI: {ami:.4f}")
    print(f"Homogeneity: {homogeneity:.4f}")
    print(f"V-measure: {v_measure:.4f}")
    print(f"Silhouette: {sil:.4f}")
    return {"ARI": ari, "NMI": nmi, "AMI": ami, "Homogeneity": homogeneity, "V-measure": v_measure, "Silhouette": sil}

y_true = adata_rna.obs['ground_truth']
kmeans_res = evaluate_clustering(y_true, adata_rna.obs['kmeans_clusters'], joint_feat, "KMeans")
leiden_res = evaluate_clustering(y_true, adata_rna.obs['leiden_clusters'], joint_feat, "Leiden (KNN graph-based)")

In [ ]:
# 9. UMAP and Spatial Visualization
print("Computing UMAP projections...")
sc.tl.umap(adata_rna)

fig, axes = plt.subplots(3, 2, figsize=(15, 18))

# Plot Ground Truth
sc.pl.umap(adata_rna, color='ground_truth', ax=axes[0, 0], title='UMAP: Ground Truth', show=False, size=20)
sc.pl.embedding(adata_rna, basis='spatial', color='ground_truth', ax=axes[0, 1], title='Spatial: Ground Truth', show=False, size=25)

# Plot KMeans
sc.pl.umap(adata_rna, color='kmeans_clusters', ax=axes[1, 0], title='UMAP: KMeans Clusters', show=False, size=20)
sc.pl.embedding(adata_rna, basis='spatial', color='kmeans_clusters', ax=axes[1, 1], title='Spatial: KMeans Clusters', show=False, size=25)

# Plot Leiden
sc.pl.umap(adata_rna, color='leiden_clusters', ax=axes[2, 0], title='UMAP: Leiden Clusters (KNN-based)', show=False, size=20)
sc.pl.embedding(adata_rna, basis='spatial', color='leiden_clusters', ax=axes[2, 1], title='Spatial: Leiden Clusters (KNN-based)', show=False, size=25)

plt.tight_layout()
plt.show()